In [ ]:
from supabase import create_client
import pandas as pd
import os
from dotenv import load_dotenv
import plotly.express as px

load_dotenv()

supabase = create_client(
  os.getenv("SUPABASE_URL"), 
  os.getenv("SUPABASE_KEY")
)

def load_table(table_name):
  response = supabase.table(table_name).select("*").execute()
  return pd.DataFrame(response.data)

sales_df = load_table("sales")
customers_df = load_table("customers")

df = pd.merge(
    sales_df,
    customers_df,
    on="customer_id",
    how="left"
)


In [ ]:
df = df.dropna(subset=["customer_id", "sale_date", "total_price"])
df["sale_date"] = pd.to_datetime(df["sale_date"])
df = df[df["total_price"] > 0]
print("Final shape:", df.shape)
print("Unique customers:", df["customer_id"].nunique())
print("Date range:", df["sale_date"].min(), "to", df["sale_date"].max())

In [ ]:
reference_date = df["sale_date"].max() + pd.Timedelta(days=1)
last_purchase = df.groupby("customer_id")["sale_date"].max()
recency = (reference_date - last_purchase).dt.days
frequency = df.groupby("customer_id")["sale_id"].count()
monetary = df.groupby("customer_id")["total_price"].sum()
rfm = pd.DataFrame({
    "recency_days": recency,
    "frequency": frequency,
    "monetary_value": monetary
})
rfm["R_score"] = pd.qcut(
    rfm["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1]
)
rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm["M_score"] = pd.qcut(
    rfm["monetary_value"],
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm["RFM_Score"] = (
    rfm["R_score"].astype(int)
    + rfm["F_score"].astype(int)
    + rfm["M_score"].astype(int)
)
def assign_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"
rfm["Segment"] = rfm["RFM_Score"].apply(assign_segment)
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="customer_count")
)
segment_summary["share"] = (
    segment_summary["customer_count"] / len(rfm) * 100
)
segment_summary["share"] = segment_summary["share"].round(1)

In [104]:
print(segment_summary)

         Segment  customer_count  share
0      Potential             759   29.9
1          Loyal             679   26.7
2        At Risk             529   20.8
3  VIP Champions             455   17.9
4           Lost             118    4.6


In [95]:

segment_order = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

fig_segments = px.bar(
    segment_summary,
    x="Segment",
    y="customer_count",
    title="Customers by RFM Segment",
    labels={
        "Segment": "Customer Segment",
        "customer_count": "Number of Customers"
    },
    category_orders={
        "Segment": segment_order
    }
)

fig_segments.show()

In [ ]:
fig_scatter = px.scatter(
    rfm,
    x="recency_days",
    y="monetary_value",
    color="Segment",
    size="frequency",
    title="UrbanStyle Customer Segments (RFM)",
    labels={
        "recency_days": "Days Since Last Purchase",
        "monetary_value": "Total Spend (EUR)",
        "frequency": "Purchase Frequency"
    }
)

fig_scatter.show()

In [103]:
top_vips = (
    rfm[rfm["Segment"] == "VIP Champions"]
    .nlargest(10, "monetary_value")
)
top_vips_chart = top_vips.reset_index().copy()
top_vips_chart["customer_id"] = (
    top_vips_chart["customer_id"]
    .astype(int)
    .astype(str)
)
fig_top_vips = px.bar(
    top_vips_chart,
    x="monetary_value",
    y="customer_id",
    orientation="h",
    title="Top 10 VIP Customers by Total Spend",
    labels={
        "customer_id": "Customer ID",
        "monetary_value": "Total Spend (EUR)"
    },
    text="monetary_value"
)
fig_top_vips.update_layout(
    yaxis={"categoryorder": "total ascending"}
)
fig_top_vips.update_traces(
    texttemplate="€%{text:,.0f}",
    textposition="outside"
)

fig_top_vips.show()